In [ ]:
%env KAGGLE_KEY_FOLDER=MDST/RvF
!mkdir data
!export KAGGLE_CONFIG_DIR=/content/drive/MyDrive/$KAGGLE_KEY_FOLDER && wget -O - "https://raw.githubusercontent.com/MichiganDataScienceTeam/W24-RvF/main/data/download.sh" | bash -s rvf10k


!rm -r W24-RvF starter_code
!git clone -q https://github.com/MichiganDataScienceTeam/W24-RvF.git
!mv W24-RvF/starter_code .
!rm -r W24-RvF

env: KAGGLE_KEY_FOLDER=MDST/RvF
mkdir: cannot create directory ‘data’: File exists


--2026-06-11 04:40:34--  https://raw.githubusercontent.com/MichiganDataScienceTeam/W24-RvF/main/data/download.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2990 (2.9K) [text/plain]
Saving to: ‘STDOUT’

-                   100%[===================>]   2.92K  --.-KB/s    in 0s      

2026-06-11 04:40:35 (125 MB/s) - written to stdout [2990/2990]

Current Python Environment: /home/ducanh19082007/Quantitative_Finance_Self_Learning/.venv/bin/python
Traceback (most recent call last):
  File "/home/ducanh19082007/Quantitative_Finance_Self_Learning/.venv/bin/kaggle", line 5, in <module>
    from kaggle.cli import main
  File "/home/ducanh19082007/Quantitative_Finance_Self_Learning/.venv/lib/python3.12/site-packages/kaggle/__init__.py", line 3, in <module>
    f

In [ ]:
!git clone https://github.com/MichiganDataScienceTeam/RvF-Challenge.git


In [ ]:
from starter_code.dataset import RvFDataset, get_loaders
from starter_code.train import train_model, plot_performance, load_model


#again, this might be a bit of a disadvantage for me because i do not use PyTorch,
#i use Tableau, so the whole architechture of the CNN i'll change it...

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import os
import cv2
import glob
from PIL import Image

import tensorflow as tf
from keras.preprocessing.image import load_img
from keras.models import Sequential, Model
from keras.layers import Dense, Input, Flatten, Conv2D, MaxPooling2D
from sklearn.model_selection import train_test_split


In [ ]:
# load training dataset
train_dataset = RvFDataset("train", data_directory = "/content/RvF-Challenge/data/rvf10k")
# load valid/test dataset
valid_dataset = RvFDataset("valid", data_directory = "/content/RvF-Challenge/data/rvf10k")

In [ ]:
import os
from PIL import Image
import random

# Path to the training directory
train_dir = "/content/RvF-Challenge/data/rvf10k/train"

# Initialize a list to hold all images
all_train_images = []
all_label_images = []

# Loop through both 'fake' and 'real' folders
for label in ['fake', 'real']:
    folder_path = os.path.join(train_dir, label)

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        try:
            # Open and load the image
            img = Image.open(file_path).resize((128, 128)).convert('RGB')  # convert to RGB just in case
            img = np.array(img) / 255.0  # normalize to [0, 1]
            all_train_images.append(img)
            all_label_images.append(label)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

all_label_images = [0 if label == "fake" else 1 for label in all_label_images]

In [ ]:
idx = np.random.randint(len(all_train_images))

plt.imshow(all_train_images[idx])
print(all_label_images[idx])
print(len(all_train_images))
print(len(all_label_images))
plt.show()

In [ ]:
#again, this might be a bit of a disadvantage for me because i do not use PyTorch,
#i use Tableau, so the whole architechture of the CNN i'll change it...

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import os
import cv2
import glob
from PIL import Image

import tensorflow as tf
from keras.preprocessing.image import load_img
from keras.models import Sequential, Model
from keras.layers import Dense, Input, Flatten, Conv2D, MaxPooling2D
from sklearn.model_selection import train_test_split



In [ ]:
#class BasicCNN(nn.Module): # Net inherits from nn.Module
    #def __init__(self):"""Constructor for the neural network."""
        #super(BasicCNN, self).__init__()        # Call superclass constructor
        #self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1)
        #self.conv2 = nn.Conv2d(in_channels=16, out_channels=128, kernel_size=3, stride=1)
        #self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        #self.relu = nn.ReLU()
        #self.flatten = nn.Flatten()
        #self.fc = nn.Linear(3200, 10)

   # def forward(self, x):
        #z1 = self.conv1(x)
        #h1 = self.relu(z1)
        #p1 = self.pool(h1)

        #z2 = self.conv2(p1)
        #h2 = self.relu(z2)
        #p2 = self.pool(h2)

        #flat = self.flatten(p2)
        #z = self.fc(flat)

        #return z

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from keras.layers import Dense, Input, Flatten, Conv2D, MaxPooling2D
from sklearn.model_selection import train_test_split

# Convert the list of images to a NumPy array

x_train, x_test, y_train, y_test = train_test_split(all_train_images, all_label_images, test_size=0.15, stratify=all_label_images, random_state=42)


x_train = np.array(x_train)
x_test = np.array(x_test)


y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

RvF_Challenge_model = models.Sequential([
    layers.Conv2D(32, kernel_size=3, padding='same',activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D(pool_size=2), # -> (64, 64, 32)

    layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=2),# -> (32,32,64)

    layers.Conv2D(128,kernel_size=3, padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=2),# -> (16,16,128)

    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dense(128,activation='relu'),
    layers.Dense(64,activation='relu'),
    layers.Dense(2, activation='softmax')
])


RvF_Challenge_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

RvF_Challenge_model.fit(x_train, y_train, epochs = 10, batch_size=250, validation_data=(x_test,y_test))
#84% of accuracy, i only limit at 350 since i don't think my laptop is fine processing this... it is relatively an old laptop

In [ ]:
predictions = RvF_Challenge_model.predict(x_test)
print(predictions[0])

prediction_labels = predictions[0].argmax()
print(prediction_labels)

num_samples_to_plot = min(80, len(x_test)) # Ensure we don't exceed the number of available samples
true_labels = np.argmax(y_test[:num_samples_to_plot], axis = 1)
predicted_labels = np.argmax(predictions[:num_samples_to_plot], axis = 1)

correctness = 0
score = 0

plt.figure(figsize=(20, 15))
for i in range(num_samples_to_plot):
    plt.subplot(8, 10, i+1)  # 2 rows, 10 images per row
    plt.imshow(x_test[i])
    plt.title(f"Pred={predicted_labels[i]}\nTrue={true_labels[i]}", fontsize=25)
    plt.axis('off')
    if predicted_labels[i] == true_labels[i]:
        correctness += 1

print((correctness/80)*100)
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model to disk
model_save_path = 'RvF_Challenge_model.h5'
RvF_Challenge_model.save(model_save_path)
print(f'Model saved to {model_save_path}')
